Import

In [1]:
import pandas as pd
import numpy as np

Load the  file

In [5]:
df_data = pd.read_csv(r"C:\Users\umroot\Desktop\BLE Data\data\ble_data_labeled.csv")
df_data.columns

Index(['user_id', 'timestamp', 'mac_address', 'RSSI', 'power',
       'year_month_day', 'hour', 'RSSI_1', 'RSSI_2', 'RSSI_3', 'RSSI_4',
       'RSSI_5', 'RSSI_6', 'RSSI_7', 'RSSI_8', 'RSSI_9', 'RSSI_10', 'RSSI_11',
       'RSSI_12', 'RSSI_13', 'RSSI_14', 'RSSI_15', 'RSSI_16', 'RSSI_17',
       'RSSI_18', 'RSSI_19', 'RSSI_20', 'RSSI_21', 'RSSI_22', 'RSSI_23',
       'RSSI_24', 'RSSI_25', 'user', 'room', 'floor'],
      dtype='object')

In [7]:
df_data = df_data.drop(columns=['user_id', 'year_month_day', 'hour', 'RSSI_1', 'RSSI_2', 'RSSI_3', 'RSSI_4',
       'RSSI_5', 'RSSI_6', 'RSSI_7', 'RSSI_8', 'RSSI_9', 'RSSI_10', 'RSSI_11',
       'RSSI_12', 'RSSI_13', 'RSSI_14', 'RSSI_15', 'RSSI_16', 'RSSI_17',
       'RSSI_18', 'RSSI_19', 'RSSI_20', 'RSSI_21', 'RSSI_22', 'RSSI_23',
       'RSSI_24', 'RSSI_25', 'user', 'floor'], errors='ignore')

In [8]:
df_data = df_data.rename(columns={'room':'location'})
df_data['location'].unique()

array(['kitchen', 'cafeteria', 'cleaning', 'nurse station', 'hallway',
       '501', '506', '520', '511', '523', '516', '508', 'Office Large',
       'Office Small', '202', 'Nurse Room', 'Cafeteria C', '213',
       'Cafeteria A', '206', '212', '211', '210', '208', '207',
       'Cafeteria D', '512', 'Cafeteria B', '522', 'Clean Room', '502',
       '518', '513', '201', 'Bathroom', 'Clean 9', '203', '517', '515',
       '503', '205', '505'], dtype=object)

Removing the rooms (cleaning "location")

In [9]:
rooms_to_remove = ['201', '202', '203', '206', '207', '208', '210', '211', '212', '213', 'Office Large', 'Office Small']
df_data = df_data[~df_data['location'].isin(rooms_to_remove)]
df_data['location'] = df_data['location'].replace({'Nurse Room': 'nurse station', 
                                       'Cafeteria A': 'cafeteria', 
                                       'Cafeteria B': 'cafeteria', 
                                       'Cafeteria C': 'cafeteria',
                                       'Cafeteria D': 'cafeteria',
                                       'Clean Room': 'cleaning',
                                       'Clean 9': 'cleaning'})

In [12]:
labels = df_data['location'].unique()
print(f"Number of labels: {len(labels)}")
print(f"Dataset shape (rows, cols): {df_data.shape}")
from IPython.display import display; display(list(df_data.columns)); display(df_data.head(5))

Number of labels: 23
Dataset shape (rows, cols): (1470478, 5)


['timestamp', 'mac_address', 'RSSI', 'power', 'location']

,timestamp,mac_address,RSSI,power,location
0,2023-04-10 14:21:46.003,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
1,2023-04-10 14:21:46.008,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
2,2023-04-10 14:21:46.012,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
3,2023-04-10 14:21:46.018,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
4,2023-04-10 14:21:46.024,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen


In [18]:
# ----------------------------------------------------
# Sliding-window Feature Engineering (10s window, 5s step)
# Spec: counts 1..25, top3 as IDs (3 columns), stats filled with 0, location right-most
# ----------------------------------------------------
from pathlib import Path

mac_list = [
    'F7:7F:78:76:7E:F3','C6:CD:5E:3D:2F:BB','D6:F4:3A:79:74:63',
    'C9:17:55:E2:3E:0E','CA:60:AB:EE:EC:7F','D6:51:7F:AB:0E:29',
    'CC:54:33:F6:A7:90','EB:20:56:87:04:5A','EE:E7:46:DC:19:6F',
    'C8:5B:BF:37:07:A0','D7:26:F6:A3:44:D2','DD:83:B0:27:FD:36',
    'E5:CD:4A:36:87:06','DC:22:B8:17:4E:B5','EA:09:20:80:D6:44',
    'E6:99:D1:EC:C6:81','F6:DA:97:C7:D5:28','EA:66:A1:12:2C:F4',
    'C9:EA:57:8B:0F:80','D6:7C:1D:2C:2A:0A','DA:E1:70:5F:44:97',
    'DD:10:10:F6:4F:27','E6:F3:93:A8:9E:22','E6:60:05:1F:88:F9',
    'D4:33:FD:F4:C2:A8'
 ]
beacon_ids = list(range(1, len(mac_list) + 1))
mac_to_id = dict(zip(mac_list, beacon_ids))

def _l2_norm(values: pd.Series) -> float:
    arr = values.to_numpy(dtype=float, copy=False)
    return float(np.linalg.norm(arr))

def make_sliding_window_features(
    df: pd.DataFrame,
    *,
    window_seconds: int = 10,
    step_seconds: int = 5,
    timestamp_col: str = 'timestamp',
    mac_col: str = 'mac_address',
    rssi_col: str = 'RSSI',
    location_col: str = 'location',
    include_empty_windows: bool = True,
 ) -> pd.DataFrame:
    window_td = pd.Timedelta(seconds=window_seconds)
    step_td = pd.Timedelta(seconds=step_seconds)
    if window_td <= pd.Timedelta(0) or step_td <= pd.Timedelta(0):
        raise ValueError('window_seconds and step_seconds must be positive')
    if window_td % step_td != pd.Timedelta(0):
        raise ValueError('window_seconds must be a multiple of step_seconds')
    required_cols = [timestamp_col, mac_col, rssi_col, location_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    x = df[required_cols].copy()
    x[timestamp_col] = pd.to_datetime(x[timestamp_col], errors='coerce')
    x = x.dropna(subset=[timestamp_col])
    x = x.sort_values(timestamp_col, kind='stable')

    # Build overlapping windows by expanding each row to all window_starts that include it.
    # For 10s window and 5s step, each row belongs to 2 windows (its floor and one previous).
    base = x[timestamp_col].dt.floor(step_td)
    n_offsets = int(window_td / step_td)  # e.g. 10/5 = 2
    parts = []
    for i in range(n_offsets):
        ws = base - (i * step_td)
        temp = x.copy()
        temp['window_start'] = ws
        temp = temp[temp[timestamp_col] < (ws + window_td)]
        parts.append(temp)
    expanded = pd.concat(parts, ignore_index=True)

    # Window index (chronological)
    if include_empty_windows and not expanded.empty:
        win_start = expanded['window_start'].min()
        win_end = expanded['window_start'].max()
        window_index = pd.date_range(start=win_start, end=win_end, freq=step_td, name='window_start')
    else:
        window_index = pd.Index(sorted(expanded['window_start'].unique()), name='window_start')
    out = pd.DataFrame(index=window_index)

    # Known beacons subset with numeric IDs (1..25)
    known = expanded[expanded[mac_col].isin(mac_to_id)].copy()
    known['beacon_id'] = known[mac_col].map(mac_to_id)

    # 1) Count features count_1..count_25
    counts = (
        known.groupby(['window_start', 'beacon_id'], sort=False)
        .size()
        .unstack(fill_value=0)
        .reindex(index=window_index, columns=beacon_ids, fill_value=0)
        .astype(int)
    )
    counts.columns = [f"count_{i}" for i in counts.columns]
    out = out.join(counts)

    # 4) Beacon variety (ALL beacons)
    out['unique_beacon_count'] = (
        expanded.groupby('window_start')[mac_col].nunique().reindex(window_index).fillna(0).astype(int)
    )

    # 5) Location label (most frequent)
    if not expanded.empty:
        loc_counts = expanded.groupby(['window_start', location_col]).size()
        loc_idx = loc_counts.groupby(level=0).idxmax()
        loc = pd.Series([t[1] for t in loc_idx], index=[t[0] for t in loc_idx])
        out[location_col] = loc.reindex(window_index).fillna('')
    else:
        out[location_col] = ''

    # 2) Top-3 strongest from top-6 frequent (known beacons only)
    count_cols = [f"count_{i}" for i in beacon_ids]
    count_arr = out[count_cols].to_numpy(dtype=int, copy=False)  # shape: (n_windows, 25)
    rssi_max = (
        known.groupby(['window_start', 'beacon_id'])[rssi_col]
        .max()
        .unstack()
        .reindex(index=window_index, columns=beacon_ids)
    )
    rssi_arr = rssi_max.to_numpy(dtype=float, copy=False)
    rssi_arr = np.where(np.isfinite(rssi_arr), rssi_arr, -np.inf)

    if count_arr.shape[0] > 0:
        top6_unsorted = np.argpartition(-count_arr, kth=min(5, count_arr.shape[1] - 1), axis=1)[:, :6]
        top6_counts = np.take_along_axis(count_arr, top6_unsorted, axis=1)
        order6 = np.argsort(-top6_counts, axis=1, kind='stable')
        top6_idx0 = np.take_along_axis(top6_unsorted, order6, axis=1)
        cand_counts = np.take_along_axis(count_arr, top6_idx0, axis=1)
        cand_rssi = np.take_along_axis(rssi_arr, top6_idx0, axis=1)
        cand_rssi = np.where(cand_counts > 0, cand_rssi, -np.inf)
        order3 = np.argsort(-cand_rssi, axis=1, kind='stable')[:, :3]
        top3_idx0 = np.take_along_axis(top6_idx0, order3, axis=1)
        top3_rssi = np.take_along_axis(cand_rssi, order3, axis=1)
        top3_valid = np.isfinite(top3_rssi)
        top3_ids = (top3_idx0 + 1).astype(int)
        top3_ids = np.where(top3_valid, top3_ids, 0).astype(int)
    else:
        top3_idx0 = np.empty((0, 3), dtype=int)
        top3_valid = np.empty((0, 3), dtype=bool)
        top3_ids = np.empty((0, 3), dtype=int)

    # Split top-3 into three columns
    out['first_beacon'] = top3_ids[:, 0] if len(out) else []
    out['second_beacon'] = top3_ids[:, 1] if len(out) else []
    out['third_beacon'] = top3_ids[:, 2] if len(out) else []

    # 3) Stats for top-3 beacons; if no samples -> 0
    stats = (
        known.groupby(['window_start', 'beacon_id'])[rssi_col]
        .agg(mean='mean', std='std', max='max', min='min', norm=_l2_norm)
        .unstack('beacon_id')
        .reindex(index=window_index)
    )
    def _gather(stat_name: str, pos: int) -> np.ndarray:
        mat = stats[stat_name].reindex(columns=beacon_ids).to_numpy(dtype=float, copy=False)
        row_idx = np.arange(mat.shape[0])
        col_idx0 = top3_idx0[:, pos]
        values = mat[row_idx, col_idx0]
        values = np.where(top3_valid[:, pos], values, 0.0)
        return np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)

    if len(out):
        out['b1_mean'] = _gather('mean', 0)
        out['b1_std'] = _gather('std', 0)
        out['b1_max'] = _gather('max', 0)
        out['b1_min'] = _gather('min', 0)
        out['b1_norm'] = _gather('norm', 0)

        out['b2_mean'] = _gather('mean', 1)
        out['b2_std'] = _gather('std', 1)
        out['b2_max'] = _gather('max', 1)
        out['b2_min'] = _gather('min', 1)
        out['b2_norm'] = _gather('norm', 1)

        out['b3_mean'] = _gather('mean', 2)
        out['b3_std'] = _gather('std', 2)
        out['b3_max'] = _gather('max', 2)
        out['b3_min'] = _gather('min', 2)
        out['b3_norm'] = _gather('norm', 2)
    else:
        for col in [
            'b1_mean','b1_std','b1_max','b1_min','b1_norm',
            'b2_mean','b2_std','b2_max','b2_min','b2_norm',
            'b3_mean','b3_std','b3_max','b3_min','b3_norm',
        ]:
            out[col] = 0.0

    # Final column order (location must be right-most)
    ordered_cols = (
        [f"count_{i}" for i in beacon_ids]
        + ['first_beacon', 'second_beacon', 'third_beacon']
        + [
            'b1_mean','b1_std','b1_max','b1_min','b1_norm',
            'b2_mean','b2_std','b2_max','b2_min','b2_norm',
            'b3_mean','b3_std','b3_max','b3_min','b3_norm',
        ]
        + ['unique_beacon_count', location_col]
    )

    out = out.sort_index()
    out = out[ordered_cols].reset_index(drop=True)
    numeric_cols = [c for c in out.columns if c != location_col]
    out[numeric_cols] = out[numeric_cols].fillna(0)
    return out

df_windows = make_sliding_window_features(df_data)
display(df_windows.head())
print('Windowed feature dataframe shape:', df_windows.shape)

# Save to CSV in data/
out_path = Path('data') / 'BLE_FE.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
df_windows.to_csv(out_path, index=False)
print('Saved:', out_path.resolve())

,count_1,count_2,count_3,count_4,count_5,count_6,count_7,count_8,count_9,count_10,...,b2_max,b2_min,b2_norm,b3_mean,b3_std,b3_max,b3_min,b3_norm,unique_beacon_count,location
0,0,0,0,214,0,68,17,0,68,0,...,-93.0,-101.0,796.197212,-94.0,1.007435,-93.0,-95.0,775.187719,9,kitchen
1,0,0,0,214,0,68,17,0,68,0,...,-93.0,-101.0,796.197212,-94.0,1.007435,-93.0,-95.0,775.187719,9,kitchen
2,0,0,0,68,0,0,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,7,cafeteria
3,0,0,0,68,0,0,0,0,0,0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,9,cafeteria
4,0,0,0,136,17,0,34,0,47,0,...,-92.0,-95.0,640.694935,-95.0,3.045115,-92.0,-98.0,554.216564,11,cafeteria


Windowed feature dataframe shape: (50720, 45)
Saved: C:\Users\umroot\Desktop\BLE Data\scripts\data\BLE_FE.csv
